# Halden Energy — Week 8: OPEC+ and Integrated Response
**Python analysis notebook (parallel to the Excel workbook)**

OPEC+ announces a production cut. Its value depends entirely on whether it holds. You estimate the probability from compliance history, then position Halden across its segments.

The integration lesson: one shock pushes four segments in different directions. If the cut holds, crude rises — upstream celebrates, refining margins compress, retail barely moves. The right answer for Halden is not the sum of the right answers for its segments.

In [ ]:
import pandas as pd
sc = pd.read_csv('data/opec_scenarios.csv')
co = {r['channel']: r['coefficient'] for _,r in pd.read_csv('data/propagation_coefficients.csv').iterrows()}
b  = {r['parameter']: r['value'] for _,r in pd.read_csv('data/baseline_state.csv').iterrows()}
ch = pd.read_csv('data/compliance_history.csv')
sc

## Part 1 — the propagation coefficients

A $1 move in WTI reaches each segment through a fixed coefficient. Upstream 1:1 (price taker); refining crack compresses at -0.35 (short-run lag); retail is nearly immune (inelastic fuel demand).

In [ ]:
def propagate(dwti):
    upstream = co['upstream_realization'] * dwti
    crack    = b['crack_base'] + co['refining_crack'] * dwti
    retail_vol_pct = co['retail_demand_elasticity'] * (co['retail_passthrough'] * dwti / 42.0 / b['pump_base']) * 100
    return upstream, crack, retail_vol_pct

for _, row in sc.iterrows():
    up, crack, vol = propagate(row['delta_wti'])
    print(f"{row['scenario']:14s} dWTI={row['delta_wti']:+5.0f} | upstream {up:+6.1f}/bbl | crack {crack:5.2f} | retail {vol:+.2f}%")
print()
print('If the cut holds full: upstream +$14/bbl but crack -$4.90. Upstream and refining root for opposite outcomes.')

## Part 2 — expected WTI and the compliance basis

You estimate the probabilities; the sim's true weights are shown for reference. Compliance history is the basis for a defensible estimate.

In [ ]:
ev = (sc['probability'] * sc['wti_resolved']).sum()
print(f'Probability-weighted expected WTI (sim weights): ${ev:.2f}')
print()
print('Compliance history (higher when the cut is deep and the market tight; collapses under fiscal pressure to cheat):')
print(ch.to_string(index=False))

## Part 3 — expected segment impact

Combine your probability estimate with the propagation coefficients. Watch the downside scenario separately — a large adverse case can matter more than its probability suggests.

In [ ]:
# Using the sim's weights as an illustration; you substitute your own estimate.
exp_upstream = sum(row['probability'] * propagate(row['delta_wti'])[0] for _,row in sc.iterrows())
exp_crack    = sum(row['probability'] * propagate(row['delta_wti'])[1] for _,row in sc.iterrows())
print(f'Expected upstream impact: {exp_upstream:+.2f}/bbl')
print(f'Expected crack:           {exp_crack:.2f}  (vs base {b["crack_base"]})')
print()
print('Upstream expects to gain; refining expects margin compression. The integration conflict is real.')

## Part 4 — your analysis

Fill the `# TODO` cells.

In [ ]:
# TODO: estimate the probability of each scenario from the compliance history (must sum to 1.0)
# your_probs = {'holds_full': None, 'holds_partial': None, 'fails': None}

# TODO: compute the expected upstream impact and crack under YOUR probabilities

# TODO: state which segment wants the cut to hold and which is hurt, and set your
#       production / run-rate / hedge posture accordingly. Does your hedge posture
#       contradict your probability estimate? (If you think the cut holds, don't hedge away the upside.)

## Your decision

- Your probability estimate, and the compliance-history reasoning behind it.
- The expected impact on each segment.
- Which segment's interest you chose to favor, and your production / run-rate / hedge posture.

The common error: a probability estimate and an operating decision that point in opposite directions — estimating the cut will hold, then hedging away the upside you just argued was likely. Keep them consistent.